[◀ Notebook 16: Modules & Packages](16_modules_and_packages.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 18: Testing & Quality ▶](18_testing_and_code_quality.ipynb)

# Notebook 17: Modern Packaging, Dependencies & Build Systems

Welcome to Notebook 17 of the Bottom-Up Python Tutorial. In this notebook, we explore the modern standards of Python packaging. We will cover the shift from legacy `setup.py` scripts to standard-compliant `pyproject.toml` files, compare Source Distributions (sdist) with Built Distributions (Wheels), examine dependency locking, and learn about the high-performance package manager `uv`.

### References:
- [Python Packaging User Guide](https://packaging.python.org/)
- [PEP 518: Specifying Build-System Requirements](https://peps.python.org/pep-0518/)
- [PEP 517: A build-system independent format for source trees](https://peps.python.org/pep-0517/)
- [PEP 621: Storing project metadata in pyproject.toml](https://peps.python.org/pep-0621/)
- [PEP 427: The Wheel Binary Package Format 1.0](https://peps.python.org/pep-0427/)

## 1. The Evolution of Python Packaging

Historically, Python packages were defined using a `setup.py` file powered by `setuptools`:
- **The Problem:** `setup.py` is an executable script. Resolving dependencies or reading metadata required running arbitrary Python code. This posed security risks, made builds non-reproducible, and was slow.
- **The Solution:** Standardize declaration format into a static, declarative configuration file: `pyproject.toml`.

### Key PEPs Shaping Modern Packaging:
1. **PEP 518 (`[build-system]`):** Allows specifying which build tools (e.g., `setuptools`, `poetry-core`, `hatchling`) are required to build the project. Python will automatically download and isolate these build requirements in a temporary virtual environment before compiling the package.
2. **PEP 517:** Standardizes the API between build frontends (like `pip` or `build`) and build backends (like `hatchling` or `flit`).
3. **PEP 621 (`[project]`):** Defines a standard, backend-agnostic layout for package metadata (e.g., name, version, authors, and dependencies) inside the `pyproject.toml` file.

## 2. Standard `pyproject.toml` Anatomy

A standard `pyproject.toml` contains three major sections:
1. `[build-system]`: Specifies the build backend requirements.
2. `[project]`: Contains metadata like name, version, dependencies, etc.
3. `[project.scripts]`: Defines command-line scripts (entry points) exposed by the package upon installation.

```toml
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "my-awesome-app"
version = "0.1.0"
description = "A CLI application to perform math."
readme = "README.md"
requires-python = ">=3.8"
dependencies = [
    "requests>=2.28.0",
]

[project.scripts]
my-app = "my_awesome_app.cli:main"
```

## 3. Distribution Formats: Wheel vs. sdist

When you package a Python project, you typically build two formats:
1. **Source Distribution (sdist):** A `.tar.gz` archive containing the original source files, templates, and build configuration. Installing from an sdist requires compiling code (if there are C extensions) and running build hooks on the user's machine.
2. **Built Distribution (Wheel):** A `.whl` archive, which is a specialized zip file containing files ready to be copied directly into the target `site-packages` directory. It requires no compilation, making installation incredibly fast and robust.

### Packaging with `uv`
[`uv`](https://github.com/astral-sh/uv) is a blazing-fast Python package installer and resolver written in Rust. It functions as a drop-in replacement for `pip`, `pip-tools`, `virtualenv`, and includes a compiler toolchain wrapper that can build packages via `uv build`.

## 4. Programmatic CLI Execution using `subprocess.run()`

While packaging and compiling code enables distributing clean command-line tools, developers often need to invoke these CLI tools, shell commands, or external executables programmatically from within Python. Python provides the `subprocess` module to manage child process creation.

### High-Level Process Management: `subprocess.run()`

Introduced in Python 3.5, `subprocess.run()` is the recommended high-level entry point for executing synchronous commands. It spawns a child process, waits for it to finish, and returns a `CompletedProcess` instance. This API replaces older, less secure interfaces such as `os.system()` and `os.spawn*()`.

#### Key Parameters for Process Execution:
- **`args`**: A sequence of strings (typically a list like `["python", "-m", "build"]`). Pass a list rather than a raw string because the shell parser can be unpredictable and poses security vulnerabilities. Using a sequence ensures Python correctly escapes and passes arguments directly to the OS.
- **`capture_output=True`**: Captures stdout and stderr automatically. The captured bytes can be accessed via `result.stdout` and `result.stderr`.
- **`text=True` (or `universal_newlines=True`)**: Configures the streams to decode captured byte arrays into Python strings (`str`) automatically, using the system-default locale encoding (e.g., UTF-8).
- **`check=True`**: Automatically raises a `subprocess.CalledProcessError` if the child process returns a non-zero exit status code, saving you from writing manual status checks.
- **`shell=True`**: Executes the command via the host operating system's command interpreter shell (e.g., `/bin/sh` on Unix or `cmd.exe` on Windows). **Warning:** Avoid `shell=True` when executing commands with untrusted user input, as it exposes the program to severe shell injection vulnerabilities.

### Under the Hood: CPython Process Spawning

How does Python spawn a child process? Under the CPython implementation:
1. **OS System Calls**: On POSIX compliance systems (such as Linux or macOS), CPython uses the standard `fork()` and `execve()` system calls (or `posix_spawn()` for performance optimizations) within the `_posixsubprocess.c` extension module. On Windows platforms, CPython invokes the Win32 API function `CreateProcessW()` inside the `_subprocess.c` C module.
2. **Blocking Flow**: When `subprocess.run()` is invoked, CPython halts the executing Python thread and waits (using POSIX `waitpid` or Windows `WaitForSingleObject`) until the spawned process exits. Once resolved, it reads the outputs from the standard stream file descriptors/handles, cleans up resources, and populates the `CompletedProcess` metadata.

### Concrete Code Example: Capturing Subprocess Execution

The code below demonstrates invoking Python's package compiler synchronously, capturing its output, checking for failures, and handling runtime errors:

```python
import subprocess
import sys

def compile_package_cli(package_dir, output_dir):
    # Command list: ['python', '-m', 'build', 'path/to/pkg', '--outdir', 'path/to/dist']
    command = [
        sys.executable,
        "-m",
        "build",
        package_dir,
        "--outdir",
        output_dir
    ]
    
    try:
        # Execute command synchronously, capture text output, and check exit status
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=True
        )
        print("Build completed successfully!")
        print("Stdout:", result.stdout)
        
    except subprocess.CalledProcessError as e:
        print(f"Process failed with return code {e.returncode}!", file=sys.stderr)
        print(f"Stderr output: {e.stderr}", file=sys.stderr)
        raise
    except FileNotFoundError:
        print("Error: The executable or python command could not be located.", file=sys.stderr)
        raise
```

---

## 5. Coding Challenges

Complete the two packaging challenges below.

### Challenge 1: Creating a Standards-Compliant `pyproject.toml`

Write a valid `pyproject.toml` to a project folder `assets/my_app_pkg/`. The file must contain:
- A `[build-system]` section with `requires = ["hatchling"]` and `build-backend = "hatchling.build"`.
- A `[project]` section with:
  - `name = "my_app"`
  - `version = "0.1.0"`
  - `dependencies` including `requests>=2.28.0` and `click>=8.0.0`.
- A `[project.scripts]` section defining a console script `my-app` that points to `my_app.main:cli_run`.

In [ ]:
import os

# First, let's create the project directory structure
os.makedirs("assets/my_app_pkg/my_app", exist_ok=True)

with open("assets/my_app_pkg/my_app/__init__.py", "w") as f:
    f.write("")

with open("assets/my_app_pkg/my_app/main.py", "w") as f:
    f.write("def cli_run():\n    print('Hello from the packaged CLI!')\n")

# TODO: Write the contents of pyproject.toml as a string and save it inside assets/my_app_pkg/pyproject.toml
pyproject_content = """
# Write your TOML content here
"""

with open("assets/my_app_pkg/pyproject.toml", "w") as f:
    f.write(pyproject_content)

In [ ]:
# Solution Code for Challenge 1
# pyproject_content = """
# [build-system]
# requires = ["hatchling"]
# build-backend = "hatchling.build"
# 
# [project]
# name = "my_app"
# version = "0.1.0"
# dependencies = [
#     "requests>=2.28.0",
#     "click>=8.0.0"
# ]
# 
# [project.scripts]
# my-app = "my_app.main:cli_run"
# """
# with open("assets/my_app_pkg/pyproject.toml", "w") as f:
#     f.write(pyproject_content)

In [ ]:
# Challenge 1 Verification Test
def check_toml(toml_str):
    try:
        import tomllib
        return tomllib.loads(toml_str)
    except ImportError:
        try:
            import tomli
            return tomli.loads(toml_str)
        except ImportError:
            # Fallback simple parser
            data = {}
            current_section = None
            for line in toml_str.splitlines():
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                if line.startswith('[') and line.endswith(']'):
                    current_section = line[1:-1].strip()
                    data[current_section] = {}
                elif '=' in line and current_section:
                    k, v = line.split('=', 1)
                    k = k.strip()
                    v = v.strip().strip('"').strip("'")
                    if v.startswith('[') and v.endswith(']'):
                        v = [item.strip().strip('"').strip("'") for item in v[1:-1].split(',') if item.strip()]
                    data[current_section][k] = v
            return data

with open("assets/my_app_pkg/pyproject.toml", "r") as f:
    content = f.read()

toml_data = check_toml(content)

assert "build-system" in toml_data, "Missing [build-system] section"
assert toml_data["build-system"].get("build-backend") == "hatchling.build", "build-backend must be hatchling.build"
assert "project" in toml_data, "Missing [project] section"
assert toml_data["project"].get("name") == "my_app", "Project name must be 'my_app'"
assert toml_data["project"].get("version") == "0.1.0", "Project version must be '0.1.0'"

deps = toml_data["project"].get("dependencies", [])
assert any("requests>=2.28.0" in d for d in deps), "Missing requests dependency requirement"
assert any("click>=8.0.0" in d for d in deps), "Missing click dependency requirement"

assert "project.scripts" in toml_data, "Missing [project.scripts] section"
assert toml_data["project.scripts"].get("my-app") == "my_app.main:cli_run", "my-app entry point must point to my_app.main:cli_run"
print("Challenge 1: Success! pyproject.toml matches specification.")

### Challenge 2: Building Distribution Archives

Using Python's standard execution tools (like `subprocess`), compile your `my_app_pkg` project into source distribution (`.tar.gz`) and Wheel (`.whl`) formats. Store the built archives inside `assets/my_app_pkg/dist`.

You can invoke the standard build library `build` via a subprocess: `python -m build my_app_pkg --outdir assets/my_app_pkg/dist` or use the command line build tools like `uv build --package`.

In [ ]:
import subprocess
import sys

# TODO: Write python code to build the package using a subprocess.
# If using pip/build, you might need to run: [sys.executable, "-m", "build", "assets/my_app_pkg", "--outdir", "assets/my_app_pkg/dist"]
# Alternatively, if uv is installed: ["uv", "build", "--out-dir", "assets/my_app_pkg/dist", "assets/my_app_pkg"]

# run the build command here
build_command = [sys.executable, "-m", "pip", "install", "build"] # Make sure build package is installed
subprocess.run(build_command)

# Now run the build itself:

In [ ]:
# Solution Code for Challenge 2
# import subprocess
# import sys
# subprocess.run([sys.executable, "-m", "pip", "install", "build"])
# subprocess.run([sys.executable, "-m", "build", "assets/my_app_pkg", "--outdir", "assets/my_app_pkg/dist"])

In [ ]:
# Challenge 2 Verification Test
import glob

dist_files = glob.glob("assets/my_app_pkg/dist/*")
print("Generated Files in dist:", dist_files)

has_wheel = any(f.endswith(".whl") for f in dist_files)
has_sdist = any(f.endswith(".tar.gz") for f in dist_files)

assert has_wheel, "Wheel distribution file (*.whl) not found in dist/"
assert has_sdist, "Source distribution file (*.tar.gz) not found in dist/"
print("Challenge 2: Success! Package compiled successfully into wheel and sdist formats.")

[◀ Notebook 16: Modules & Packages](16_modules_and_packages.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 18: Testing & Quality ▶](18_testing_and_code_quality.ipynb)